# Qa Chain Handles Empty Response

> **Source:** `repo1/testing_patterns.py`

Test chain handles empty responses.


## Imports and Setup


In [ ]:
import pytest
from unittest.mock import Mock, patch
from typing import Callable
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langsmith import traceable, Client
from dotenv import load_dotenv
load_dotenv()
class QAChain:
    """Simple Q&A chain for testing."""

    def __init__(self, llm=None):
        self.llm = llm or ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.prompt = ChatPromptTemplate.from_template(
            "Answer this question: {question}"
        )

    def ask(self, question: str) -> str:
        prompt_value = self.prompt.invoke({"question": question})
        response = self.llm.invoke(prompt_value)
        return response.content
class IntegrationTestSuite:
    """Integration tests with real LLM calls."""

    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @traceable(name="integration_test")
    def test_basic_qa(self) -> dict:
        """Test basic question answering."""

        test_cases = [
            {
                "question": "What is 2 + 2?",
                "expected_contains": ["4", "four"],
            },
            {
                "question": "What color is the sky on a clear day?",
                "expected_contains": ["blue"],
            },
        ]

        results = []
        for case in test_cases:
            response = self.llm.invoke(case["question"])
            content = response.content.lower()

            passed = any(exp.lower() in content for exp in case["expected_contains"])

            # "The answer is 4" or "2 + 2 equals four" or "That would be 4."

            results.append(
                {
                    "question": case["question"],
                    "response": response.content,
                    "passed": passed,
                }
            )

        return {
            "total": len(results),
            "passed": sum(1 for r in results if r["passed"]),
            "results": results,
        }
class LLMEvaluator:
    """Use LLM to evaluate LLM outputs."""

    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @traceable(name="evaluate_response")
    def evaluate(self, question: str, response: str, reference: str = None) -> dict:
        """Evaluate a response on multiple dimensions."""

        eval_prompt = ChatPromptTemplate.from_template(
            """
Evaluate this response on a scale of 1-10 for each criterion.

Question: {question}
Response: {response}
{reference_section}

Rate each criterion (1-10):
1. Correctness: Is the information accurate?
2. Relevance: Does it answer the question?
3. Clarity: Is it easy to understand?
4. Completeness: Does it fully address the question?

Respond with ONLY a JSON object:
{{"correctness": X, "relevance": X, "clarity": X, "completeness": X, "overall": X}}
"""
        )

        reference_section = ""
        if reference:
            reference_section = f"Reference answer: {reference}"

        import json

        response_obj = self.llm.invoke(
            eval_prompt.format(
                question=question,
                response=response,
                reference_section=reference_section,
            )
        )

        try:
            scores = json.loads(response_obj.content)
            return scores
        except json.JSONDecodeError:
            return {"error": "Failed to parse evaluation"}
class RegressionTestRunner:
    """Run regression tests against a test dataset."""

    def __init__(self, chain: Callable):
        self.chain = chain
        self.evaluator = LLMEvaluator()

    @traceable(name="regression_test")
    def run(self, test_cases: list[dict]) -> dict:
        """
        Run regression tests.

        test_cases: [{"input": ..., "expected": ...}, ...]
        """
        results = []
        total_score = 0

        for case in test_cases:
            # Get response from chain
            response = self.chain(case["input"])

            # Evaluate
            scores = self.evaluator.evaluate(
                question=case["input"],
                response=response,
                reference=case.get("expected"),
            )

            overall = scores.get("overall", 0)
            total_score += overall

            results.append(
                {
                    "input": case["input"],
                    "response": response,
                    "expected": case.get("expected"),
                    "scores": scores,
                    "passed": overall >= 7,  # Threshold
                }
            )

        return {
            "total": len(results),
            "passed": sum(1 for r in results if r["passed"]),
            "average_score": total_score / len(results) if results else 0,
            "results": results,
        }
from langsmith import Client
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langsmith import traceable
from dotenv import load_dotenv
load_dotenv()
client = Client()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_template("Answer this question concisely: {question}")
qa_chain = prompt | llm
eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## Implementation


In [ ]:
def test_qa_chain_handles_empty_response():
    """Test chain handles empty responses."""

    mock_llm = Mock()
    mock_llm.invoke.return_value = AIMessage(content="")

    chain = QAChain(llm=mock_llm)
    result = chain.ask("Empty question")

    assert result == ""


## Execute Demo


In [ ]:
test_qa_chain_handles_empty_response()
